# Custom components from functions

A component needs a function and initial parameter values. An analytic derivative is optional. The same interface works for generic 1D data; here we use a linear trend in arbitrary coordinate and value units. See [Line Components](Line%20Components.ipynb) to combine a custom profile with transition positions and relative weights.

Components are the building blocks of a 1D model. Combine them with arithmetic or pipe operators; calling the final model on a grid evaluates the expression in that order. This example needs no spectroscopy-specific infrastructure.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from astropy.table import Table

from prism.modeling import Component
from prism.modeling.fitting import TRF, FitResult

## Define the function

Parameter names follow the function arguments after `x`. The reference coordinate and scale below simply keep the slope near unity.

Derivative rows follow the declared parameter order.

In [ ]:
def continuum(x, amplitude, slope):
    return amplitude + slope*(x-4950.0)/100.0


def continuum_deriv(x, amplitude, slope):
    return np.array([np.ones_like(x), (x-4950.0)/100.0])

In [ ]:
component = Component(
    continuum, {"amplitude": 1.0, "slope": 0.0},
    derivative=continuum_deriv, name="component")
component.slope.bounds = (-1.0, 1.0)
component

Attributes and `params` refer to the same live parameters. The mapping is useful when names come from a configuration or a loop. Omitting `derivative` lets the fitter use finite differences.

In [ ]:
component.params["amplitude"].bounds = (0.0, None)
{name: parameter.value for name, parameter in component.params.items()}

## Fit synthetic data

Generate noisy data independently of the initial component. Errors are independent Gaussian standard deviations.

In [ ]:
x = np.linspace(4800, 5060, 1600)
truth = continuum(x, 1.3, 0.15)
yerr = np.full(x.size, 0.03)
y = truth + np.random.default_rng(41).normal(0, yerr)

In [ ]:
fitter = TRF()
result = fitter(component, x, y, yerr=yerr)
fitted = result.model

Table(rows=[(name, p.value, p.std) for name, p in result.params.items()],
      names=("parameter", "value", "std"))

In [ ]:
fig, axes = plt.subplots(2, 1, sharex=True, figsize=(8, 5), constrained_layout=True)
axes[0].plot(x, y, ".", color="0.7", markersize=2, label="Data")
axes[0].plot(x, result.predict(), label="Fit")
axes[0].set_ylabel("Value")
axes[0].legend()
axes[1].plot(x, (y-result.predict())/yerr, ".", markersize=2)
axes[1].axhline(0, color="0.5")
axes[1].set(xlabel="Coordinate", ylabel="Residual / error")

## Save and reuse

The result stores fitted parameters, data and covariance. Supply the function component when loading: the file stores its state, not executable Python code.

In [ ]:
from pathlib import Path

Path("results").mkdir(exist_ok=True)
result.save("results/custom.fits", overwrite=True)
restored = FitResult.load("results/custom.fits", components={"component": component})
restored.params

## Reuse an Astropy model

Adapt an Astropy model explicitly, then combine it with a native component. This constant is fixed so it does not introduce another free continuum normalization.

In [ ]:
from astropy.modeling.models import Const1D
from prism.modeling import Model

mixed = fitted + Model.from_astropy(Const1D(0, fixed={"amplitude": True}))
mixed